In [1]:
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE
import gdown

# =====================================================
# CREATE RESULT FOLDERS
# =====================================================
os.makedirs("Results/plots", exist_ok=True)
os.makedirs("Results/accuracy_tables", exist_ok=True)

# =====================================================
# DEVICE
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================================
# PUBLIC DRIVE FILES
# =====================================================
TEST_EMB_ID   = "1LE9cr4R2gogfh0_5Cja4k08zXLcOGjjX"
TEST_LABEL_ID = "1rSbwxM9L_Dp8LA8X1dvGg_RYeaoHqk8a"
MODEL_ID      = "1i9vO_yxtZUxA57FtrH22c-S3bODWLCDf"

# Download files if not present
os.makedirs("checkpoints", exist_ok=True)
if not os.path.exists("checkpoints/test_embeddings.pt"):
    gdown.download(f"https://drive.google.com/uc?id={TEST_EMB_ID}",
                   "checkpoints/test_embeddings.pt", quiet=False)
if not os.path.exists("checkpoints/test_labels.pt"):
    gdown.download(f"https://drive.google.com/uc?id={TEST_LABEL_ID}",
                   "checkpoints/test_labels.pt", quiet=False)
os.makedirs("models/speech_pipeline", exist_ok=True)
if not os.path.exists("models/speech_pipeline/best_model.pth"):
    gdown.download(f"https://drive.google.com/uc?id={MODEL_ID}",
                   "models/speech_pipeline/best_model.pth", quiet=False)

# =====================================================
# LOAD TEST EMBEDDINGS
# =====================================================
test_padded = torch.load("checkpoints/test_embeddings.pt")
test_labels_tensor = torch.load("checkpoints/test_labels.pt")
print("Test embeddings loaded!")

# =====================================================
# EMOTION LABELS
# =====================================================
emotion_names = ["angry","disgust","fear","happy","neutral","sad","surprise"]

# =====================================================
# ATTENTION POOLING
# =====================================================
class AttentionPooling(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.attention = nn.Linear(input_dim, 1)
    def forward(self, x):
        weights = torch.softmax(self.attention(x), dim=1)
        pooled = torch.sum(weights * x, dim=1)
        return pooled

# =====================================================
# MODEL
# =====================================================
class EmotionModel(nn.Module):
    def __init__(self, hidden_size=256, num_layers=2, num_classes=7):
        super().__init__()
        self.lstm = nn.LSTM(input_size=768, hidden_size=hidden_size, num_layers=num_layers,
                            batch_first=True, bidirectional=True, dropout=0.3)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size*2, nhead=8,
                                                   dim_feedforward=512, dropout=0.3, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.attention_pool = AttentionPooling(hidden_size*2)
        self.fc1 = nn.Linear(hidden_size*2, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        trans_out = self.transformer(lstm_out)
        pooled = self.attention_pool(trans_out)
        x = self.fc1(pooled)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.fc2(x)

# =====================================================
# LOAD MODEL
# =====================================================
model = EmotionModel().to(device)
model.load_state_dict(torch.load("models/speech_pipeline/best_model.pth"))
model.eval()
print("Model loaded successfully!")

# =====================================================
# PREDICTIONS
# =====================================================
all_preds = []
with torch.no_grad():
    for i in tqdm(range(len(test_padded))):
        sample = test_padded[i].unsqueeze(0).to(device)
        outputs = model(sample)
        pred = torch.argmax(outputs, dim=1).item()
        all_preds.append(pred)

# =====================================================
# ACCURACY & REPORT
# =====================================================
accuracy = accuracy_score(test_labels_tensor.numpy(), all_preds)
print("\nTest Accuracy:", accuracy)

report = classification_report(test_labels_tensor.numpy(), all_preds, target_names=emotion_names)
print(report)

accuracy_path = "Results/accuracy_tables/speech_accuracy_report.txt"
with open(accuracy_path, "w") as f:
    f.write(f"Test Accuracy: {accuracy}\n\n")
    f.write(report)
print(f"Accuracy report saved at:\n{accuracy_path}")

# =====================================================
# CONFUSION MATRIX
# =====================================================
cm = confusion_matrix(test_labels_tensor.numpy(), all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=emotion_names, yticklabels=emotion_names)
plt.title("Speech Emotion Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
conf_matrix_path = "Results/plots/speech_confusion_matrix.png"
plt.savefig(conf_matrix_path, dpi=300, bbox_inches="tight")
print(f"Confusion matrix saved at:\n{conf_matrix_path}")
plt.close()

# =====================================================
# TEMPORAL EMBEDDINGS + TSNE
# =====================================================
temporal_embeddings = []
with torch.no_grad():
    for i in tqdm(range(len(test_padded))):
        sample = test_padded[i].unsqueeze(0).to(device)
        lstm_out, _ = model.lstm(sample)
        trans_out = model.transformer(lstm_out)
        pooled = model.attention_pool(trans_out)
        temporal_embeddings.append(pooled.squeeze(0).cpu())
temporal_embeddings = torch.stack(temporal_embeddings)
print("Temporal embeddings extracted!")

tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(temporal_embeddings.numpy())
plt.figure(figsize=(10,8))
scatter = plt.scatter(reduced[:,0], reduced[:,1],
                      c=test_labels_tensor.numpy(), cmap="tab10")
legend1 = plt.legend(*scatter.legend_elements(), title="Emotions")
plt.gca().add_artist(legend1)
plt.title("Emotion Cluster Visualization (Temporal Modelling Block)")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
tsne_path = "Results/plots/temporal_tsne.png"
plt.savefig(tsne_path, dpi=300, bbox_inches="tight")
print(f"t-SNE plot saved at:\n{tsne_path}")
plt.close()

print("\nSpeech pipeline testing completed successfully!")

Using device: cuda


Downloading...
From (original): https://drive.google.com/uc?id=1LE9cr4R2gogfh0_5Cja4k08zXLcOGjjX
From (redirected): https://drive.google.com/uc?id=1LE9cr4R2gogfh0_5Cja4k08zXLcOGjjX&confirm=t&uuid=09647f89-6140-4105-aae5-3adad09eee31
To: /content/checkpoints/test_embeddings.pt
100%|██████████| 611M/611M [00:14<00:00, 43.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1rSbwxM9L_Dp8LA8X1dvGg_RYeaoHqk8a
To: /content/checkpoints/test_labels.pt
100%|██████████| 12.8k/12.8k [00:00<00:00, 32.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1i9vO_yxtZUxA57FtrH22c-S3bODWLCDf
From (redirected): https://drive.google.com/uc?id=1i9vO_yxtZUxA57FtrH22c-S3bODWLCDf&confirm=t&uuid=cda97478-0712-4568-a80c-660fac829bbc
To: /content/models/speech_pipeline/best_model.pth
100%|██████████| 27.9M/27.9M [00:00<00:00, 132MB/s]


Test embeddings loaded!
Model loaded successfully!


100%|██████████| 1400/1400 [00:10<00:00, 128.06it/s]



Test Accuracy: 0.7457142857142857
              precision    recall  f1-score   support

       angry       0.62      0.41      0.49       200
     disgust       0.94      0.60      0.73       200
        fear       1.00      0.89      0.94       200
       happy       0.60      0.80      0.69       200
     neutral       1.00      1.00      1.00       200
         sad       0.51      0.99      0.68       200
    surprise       1.00      0.53      0.69       200

    accuracy                           0.75      1400
   macro avg       0.81      0.75      0.74      1400
weighted avg       0.81      0.75      0.74      1400

Accuracy report saved at:
Results/accuracy_tables/speech_accuracy_report.txt
Confusion matrix saved at:
Results/plots/speech_confusion_matrix.png


100%|██████████| 1400/1400 [00:09<00:00, 143.11it/s]


Temporal embeddings extracted!
t-SNE plot saved at:
Results/plots/temporal_tsne.png

Speech pipeline testing completed successfully!


In [2]:
pip uninstall torchvision -y

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [3]:
import os
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset
import gdown, zipfile

# ==================================================
# PROJECT PATHS
# ==================================================
os.makedirs("dataset", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("models/text_pipeline", exist_ok=True)
os.makedirs("Results/logs", exist_ok=True)

# ==================================================
# PUBLIC DRIVE FILES
# ==================================================
DATASET_ID = "1F3ZSXpEifKW5brudmFKJ0lohcDS4X-Iz"   # tess.zip
FUSION_EMB_ID = "1QLq4IjYt_6dMqW54NY8_ptIeBZvJVXy0" # fusion_train_text_embeddings.pt

# === DOWNLOAD DATASET ===
if not os.path.exists("dataset.zip"):
    gdown.download(f"https://drive.google.com/uc?id={DATASET_ID}", "dataset.zip", quiet=False)
    with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("dataset")

# Auto-detect the correct TESS folder
main_folder = None
for f in os.listdir("dataset"):
    if "TESS Toronto emotional speech set data" in f:
        main_folder = os.path.join("dataset", f)
        break
if main_folder is None:
    raise FileNotFoundError("Could not find TESS dataset folder after extraction.")

print("Using dataset folder:", main_folder)

# === DOWNLOAD FUSION EMBEDDINGS ===
if not os.path.exists("checkpoints/fusion_train_text_embeddings.pt"):
    gdown.download(f"https://drive.google.com/uc?id={FUSION_EMB_ID}",
                   "checkpoints/fusion_train_text_embeddings.pt", quiet=False)

# ==================================================
# DATASET CREATION
# ==================================================
data = []
for folder in os.listdir(main_folder):
    folder_path = os.path.join(main_folder, folder)
    if os.path.isdir(folder_path):
        emotion = folder.split("_")[-1].lower()
        if emotion == "surprised":
            emotion = "surprise"
        for file in os.listdir(folder_path):
            if file.endswith(".wav"):
                word = file.split("_")[1].lower()
                data.append([word, emotion])

df = pd.DataFrame(data, columns=["text", "emotion"])
df["text"] = df["text"].apply(lambda t: t.lower().strip())

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["emotion"])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["emotion"],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=16
    )

train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

model_path = "models/text_pipeline/best_text_model"
if os.path.exists(model_path):
    print("Loading pretrained text model...")
    model = DistilBertForSequenceClassification.from_pretrained(model_path)
else:
    print("Training new text model...")
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=7
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# ==================================================
# TRAINING
# ==================================================
training_args = TrainingArguments(
    output_dir="models/text_pipeline",
    eval_strategy="epoch",   # <-- use this instead of evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="Results/logs",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print("Best text model saved at:", model_path)

Downloading...
From (original): https://drive.google.com/uc?id=1F3ZSXpEifKW5brudmFKJ0lohcDS4X-Iz
From (redirected): https://drive.google.com/uc?id=1F3ZSXpEifKW5brudmFKJ0lohcDS4X-Iz&confirm=t&uuid=72921e1c-a9b5-4753-801f-e4ef11aec4b1
To: /content/dataset.zip
100%|██████████| 449M/449M [00:07<00:00, 63.3MB/s]


Using dataset folder: dataset/TESS Toronto emotional speech set data


Downloading...
From: https://drive.google.com/uc?id=1QLq4IjYt_6dMqW54NY8_ptIeBZvJVXy0
To: /content/checkpoints/fusion_train_text_embeddings.pt
100%|██████████| 4.30M/4.30M [00:00<00:00, 16.6MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best text model saved at: models/text_pipeline/best_text_model


In [4]:

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from datasets import Dataset

# ==================================================
# PROJECT PATHS
# ==================================================
os.makedirs("Results/plots", exist_ok=True)
os.makedirs("Results/accuracy_tables", exist_ok=True)

# ==================================================
# DEVICE
# ==================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==================================================
# LOAD DATASET
# ==================================================
main_folder = None

for f in os.listdir("dataset"):
    if "TESS Toronto emotional speech set data" in f:
        main_folder = os.path.join("dataset", f)
        break

if main_folder is None:
    raise FileNotFoundError("Dataset folder not found.")

print("Using dataset folder:", main_folder)

# ==================================================
# CREATE DATAFRAME
# ==================================================
data = []

for folder in os.listdir(main_folder):
    folder_path = os.path.join(main_folder, folder)

    if os.path.isdir(folder_path):

        emotion = folder.split("_")[-1].lower()

        if emotion == "surprised":
            emotion = "surprise"

        for file in os.listdir(folder_path):

            if file.endswith(".wav"):

                word = file.split("_")[1].lower()

                data.append([word, emotion])

df = pd.DataFrame(data, columns=["text", "emotion"])

# ==================================================
# LABEL ENCODING
# ==================================================
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["emotion"])

# ==================================================
# TRAIN TEST SPLIT
# ==================================================
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["emotion"],
    random_state=42
)

test_df = test_df.reset_index(drop=True)

# ==================================================
# LOAD TOKENIZER & MODEL
# ==================================================
model_path = "models/text_pipeline/best_text_model"

tokenizer = DistilBertTokenizer.from_pretrained(model_path)

model = DistilBertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()

print("Best text model loaded successfully!")

# ==================================================
# TOKENIZE TEST DATA
# ==================================================
encodings = tokenizer(
    test_df["text"].tolist(),
    truncation=True,
    padding=True,
    max_length=16,
    return_tensors="pt"
)

input_ids = encodings["input_ids"].to(device)
attention_mask = encodings["attention_mask"].to(device)

labels = test_df["label"].tolist()

# ==================================================
# PREDICTIONS
# ==================================================
with torch.no_grad():

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    logits = outputs.logits

preds = torch.argmax(logits, dim=1).cpu().numpy()

# ==================================================
# ACCURACY
# ==================================================
accuracy = accuracy_score(labels, preds)

print("Test Accuracy:", accuracy)

# ==================================================
# CLASSIFICATION REPORT
# ==================================================
report = classification_report(labels, preds, zero_division=0)

print(report)

accuracy_path = "Results/accuracy_tables/text_results.txt"

with open(accuracy_path, "w") as f:
    f.write(f"Accuracy: {accuracy}\n\n")
    f.write(report)

print("Accuracy report saved!")

# ==================================================
# CONFUSION MATRIX
# ==================================================
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(10,8))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.title("Text Emotion Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

conf_matrix_path = "Results/plots/text_confusion_matrix.png"

plt.savefig(conf_matrix_path, bbox_inches="tight", dpi=300)

plt.close()

print("Confusion matrix saved!")

# ==================================================
# EXTRACT CLS EMBEDDINGS
# ==================================================
with torch.no_grad():

    hidden_states = model.distilbert(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

cls_embeddings = hidden_states.last_hidden_state[:,0,:].cpu().numpy()

# ==================================================
# TSNE
# ==================================================
tsne = TSNE(n_components=2, random_state=42)

reduced = tsne.fit_transform(cls_embeddings)

plt.figure(figsize=(10,8))

scatter = plt.scatter(
    reduced[:,0],
    reduced[:,1],
    c=labels,
    cmap="tab10"
)

legend1 = plt.legend(
    *scatter.legend_elements(),
    title="Emotions"
)

plt.gca().add_artist(legend1)

plt.title("Text Emotion t-SNE Visualization")

tsne_path = "Results/plots/text_tsne.png"

plt.savefig(tsne_path, bbox_inches="tight", dpi=300)

plt.close()

print("t-SNE plot saved!")

print("\nText pipeline testing completed successfully!")

Using device: cuda
Using dataset folder: dataset/TESS Toronto emotional speech set data


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Best text model loaded successfully!
Test Accuracy: 0.1392857142857143
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        80
           1       0.00      0.00      0.00        80
           2       0.14      0.95      0.25        80
           3       0.00      0.00      0.00        80
           4       0.00      0.00      0.00        80
           5       0.06      0.03      0.03        80
           6       0.00      0.00      0.00        80

    accuracy                           0.14       560
   macro avg       0.03      0.14      0.04       560
weighted avg       0.03      0.14      0.04       560

Accuracy report saved!
Confusion matrix saved!
t-SNE plot saved!

Text pipeline testing completed successfully!


In [5]:
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.manifold import TSNE
import gdown

# =====================================================
# RESULT FOLDERS
# =====================================================
os.makedirs("Results/plots", exist_ok=True)
os.makedirs("Results/accuracy_tables", exist_ok=True)

# =====================================================
# DEVICE
# =====================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =====================================================
# PATHS
# =====================================================
checkpoint_path = "checkpoints"
model_path = "models/fusion_pipeline/best_fusion_model.pth"
os.makedirs(checkpoint_path, exist_ok=True)
os.makedirs("models/fusion_pipeline", exist_ok=True)

# =====================================================
# DOWNLOAD REQUIRED FILES FROM DRIVE
# =====================================================
files = {
    "fusion_test_speech_embeddings.pt": "1BrRKgkbFFElAbgV_LfR2_48S0eSqswe1",
    "fusion_test_text_embeddings.pt": "1CkooL4yli5r1XGL6O2_vvAjPg4GD2e3e",
    "fusion_test_labels.pt": "1czavHL7Flci7MD4IfSrEQOZJQxWcMWYr",
    "best_fusion_model.pth": "1dg_dr-_ueyDWs_Jm4WeH-CrXdr7OUBLt"
}

for fname, fid in files.items():
    if fname.endswith(".pt"):
        fpath = os.path.join(checkpoint_path, fname)
    else:
        fpath = model_path
    if not os.path.exists(fpath):
        gdown.download(f"https://drive.google.com/uc?id={fid}", fpath, quiet=False)

# =====================================================
# LOAD TEST EMBEDDINGS & LABELS
# =====================================================
test_speech_embeddings = torch.load(os.path.join(checkpoint_path, "fusion_test_speech_embeddings.pt"))
test_text_embeddings   = torch.load(os.path.join(checkpoint_path, "fusion_test_text_embeddings.pt"))
test_labels_tensor     = torch.load(os.path.join(checkpoint_path, "fusion_test_labels.pt"))
labels = test_labels_tensor.tolist()

fusion_test = torch.cat([test_speech_embeddings, test_text_embeddings], dim=1)
print("Fusion test shape:", fusion_test.shape)

# =====================================================
# EMOTION LABELS
# =====================================================
emotion_names = ["angry","disgust","fear","happy","neutral","sad","surprise"]

# =====================================================
# DATASET
# =====================================================
test_dataset = TensorDataset(fusion_test, test_labels_tensor)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# =====================================================
# MODEL
# =====================================================
class FusionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 7)
        )
    def forward(self, x):
        return self.network(x)

# =====================================================
# LOAD SAVED MODEL
# =====================================================
model = FusionClassifier().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Best fusion model loaded successfully!")

# =====================================================
# PREDICTIONS
# =====================================================
preds, labels_eval = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        pred = torch.argmax(outputs, dim=1)
        preds.extend(pred.cpu().numpy())
        labels_eval.extend(batch_y.numpy())

# =====================================================
# ACCURACY & REPORT
# =====================================================
accuracy = accuracy_score(labels_eval, preds)
print("\nFusion Accuracy:", accuracy)

report = classification_report(labels_eval, preds, target_names=emotion_names)
print(report)

accuracy_path = "Results/accuracy_tables/fusion_accuracy_report.txt"
with open(accuracy_path, "w") as f:
    f.write(f"Fusion Accuracy: {accuracy}\n\n")
    f.write(report)
print(f"Accuracy report saved at:\n{accuracy_path}")

# =====================================================
# CONFUSION MATRIX
# =====================================================
cm = confusion_matrix(labels_eval, preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=emotion_names, yticklabels=emotion_names)
plt.title("Fusion Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
fusion_cm_path = "Results/plots/fusion_confusion_matrix.png"
plt.savefig(fusion_cm_path, dpi=300, bbox_inches="tight")
print(f"Fusion confusion matrix saved at:\n{fusion_cm_path}")
plt.close()

# =====================================================
# FEATURE EXTRACTION FOR TSNE
# =====================================================
fusion_features = []
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        features = model.network[:-1](batch_x)  # remove final layer
        fusion_features.append(features.cpu())
fusion_features = torch.cat(fusion_features).numpy()
print("Fusion features extracted!")

# =====================================================
# TSNE VISUALIZATION
# =====================================================
tsne = TSNE(n_components=2, random_state=42)
reduced = tsne.fit_transform(fusion_features)
plt.figure(figsize=(10,8))
scatter = plt.scatter(reduced[:,0], reduced[:,1], c=labels_eval, cmap="tab10")
legend1 = plt.legend(*scatter.legend_elements(), title="Emotions")
plt.gca().add_artist(legend1)
plt.title("Fusion Representation Emotion Clusters")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
fusion_tsne_path = "Results/plots/fusion_tsne.png"
plt.savefig(fusion_tsne_path, dpi=300, bbox_inches="tight")
print(f"Fusion t-SNE saved at:\n{fusion_tsne_path}")
plt.close()

print("\nFusion pipeline evaluation completed successfully!")

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1BrRKgkbFFElAbgV_LfR2_48S0eSqswe1
To: /content/checkpoints/fusion_test_speech_embeddings.pt
100%|██████████| 4.30M/4.30M [00:00<00:00, 12.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1CkooL4yli5r1XGL6O2_vvAjPg4GD2e3e
To: /content/checkpoints/fusion_test_text_embeddings.pt
100%|██████████| 4.30M/4.30M [00:00<00:00, 25.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1czavHL7Flci7MD4IfSrEQOZJQxWcMWYr
To: /content/checkpoints/fusion_test_labels.pt
100%|██████████| 12.9k/12.9k [00:00<00:00, 37.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1dg_dr-_ueyDWs_Jm4WeH-CrXdr7OUBLt
To: /content/models/fusion_pipeline/best_fusion_model.pth
100%|██████████| 8.44M/8.44M [00:00<00:00, 142MB/s]


Fusion test shape: torch.Size([1400, 1536])
Best fusion model loaded successfully!

Fusion Accuracy: 0.7792857142857142
              precision    recall  f1-score   support

       angry       0.65      0.77      0.70       200
     disgust       0.84      0.90      0.87       200
        fear       0.99      0.91      0.95       200
       happy       0.41      0.60      0.49       200
     neutral       0.98      0.99      0.99       200
         sad       0.95      0.98      0.97       200
    surprise       0.94      0.30      0.46       200

    accuracy                           0.78      1400
   macro avg       0.82      0.78      0.77      1400
weighted avg       0.82      0.78      0.77      1400

Accuracy report saved at:
Results/accuracy_tables/fusion_accuracy_report.txt
Fusion confusion matrix saved at:
Results/plots/fusion_confusion_matrix.png
Fusion features extracted!
Fusion t-SNE saved at:
Results/plots/fusion_tsne.png

Fusion pipeline evaluation completed successfully